In [1]:
pip install pandas requests yfinance

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import requests
import yfinance as yf
from datetime import datetime, timedelta

def extrair_dolar_bcb_olinda(dias=30):
    """Extrai a cotação do Dólar PTAX (Venda) da API Olinda do BCB."""
    hoje = datetime.today()
    inicio = hoje - timedelta(days=dias)
    
    data_final = hoje.strftime('%m-%d-%Y')
    data_inicial = inicio.strftime('%m-%d-%Y')
    
    url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicial}'&@dataFinalCotacao='{data_final}'&$format=json"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        
        dados_json = response.json()
        if 'value' not in dados_json or not dados_json['value']:
            print("Aviso: A API do Banco Central não retornou dados para este período.")
            return pd.DataFrame()
            
        df_usd = pd.DataFrame(dados_json['value'])
        
        df_usd['data'] = pd.to_datetime(df_usd['dataHoraCotacao']).dt.normalize()
        df_usd = df_usd[['data', 'cotacaoVenda']]
        df_usd.rename(columns={'data': 'Data', 'cotacaoVenda': 'Cotacao_USD_BRL'}, inplace=True)
        
        df_usd = df_usd.drop_duplicates(subset=['Data'], keep='last')
        
        return df_usd
    
    except Exception as e:
        print(f"Erro na extração do BCB (Olinda): {e}")
        return pd.DataFrame()

def extrair_brent_yfinance():
    """Extrai o preço de fechamento do Petróleo Brent via Yahoo Finance."""
    try:
        ticker = yf.Ticker("BZ=F")
        df_brent = ticker.history(period="1mo").reset_index()
        
        df_brent = df_brent[['Date', 'Close']].copy()
        df_brent.rename(columns={'Date': 'Data', 'Close': 'Preco_Brent_USD'}, inplace=True)
        
        df_brent['Data'] = pd.to_datetime(df_brent['Data']).dt.tz_localize(None).dt.normalize()
        return df_brent
    
    except Exception as e:
        print(f"Erro na extração do YFinance: {e}")
        return pd.DataFrame()

def executar_pipeline_fase1():
    print("Iniciando extração de dados (Olinda + YFinance)...")
    
    df_usd = extrair_dolar_bcb_olinda()
    df_brent = extrair_brent_yfinance()
    
    if df_usd.empty or df_brent.empty:
        print("Pipeline abortado. Falha na coleta das fontes.")
        return
    
    df_final = pd.merge(df_brent, df_usd, on='Data', how='inner')
    
    df_final['Preco_Brent_BRL'] = (df_final['Preco_Brent_USD'] * df_final['Cotacao_USD_BRL']).round(2)
    df_final['Preco_Brent_USD'] = df_final['Preco_Brent_USD'].round(2)
    
    df_final.sort_values('Data', inplace=True)
    
    print("\n✅ Transformação concluída. Amostra dos dados processados:")
    print("-" * 65)
    print(df_final.tail(5).to_string(index=False))
    print("-" * 65)

if __name__ == "__main__":
    executar_pipeline_fase1()

Iniciando extração de dados (Olinda + YFinance)...

✅ Transformação concluída. Amostra dos dados processados:
-----------------------------------------------------------------
      Data  Preco_Brent_USD  Cotacao_USD_BRL  Preco_Brent_BRL
2026-09-16           105.83           5.1527           545.31
2026-09-17           104.82           5.1521           540.04
2026-09-18           103.87           5.1575           535.71
2026-09-21           100.34           5.1117           512.91
2026-09-22            99.07           5.1161           506.85
-----------------------------------------------------------------


In [3]:
!pip install google-cloud-bigquery db-dtypes

In [4]:
import pandas as pd
import requests
import yfinance as yf
from datetime import datetime, timedelta
import json
from google.oauth2 import service_account
from google.cloud import bigquery
from kaggle_secrets import UserSecretsClient

def extrair_dolar_bcb_olinda(dias=30):
    hoje = datetime.today()
    inicio = hoje - timedelta(days=dias)
    data_final = hoje.strftime('%m-%d-%Y')
    data_inicial = inicio.strftime('%m-%d-%Y')
    url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicial}'&@dataFinalCotacao='{data_final}'&$format=json"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    response = requests.get(url, headers=headers, timeout=15)
    dados_json = response.json()
    if 'value' not in dados_json or not dados_json['value']:
        return pd.DataFrame()
        
    df_usd = pd.DataFrame(dados_json['value'])
    df_usd['data'] = pd.to_datetime(df_usd['dataHoraCotacao']).dt.normalize()
    df_usd = df_usd[['data', 'cotacaoVenda']]
    df_usd.rename(columns={'data': 'Data', 'cotacaoVenda': 'Cotacao_USD_BRL'}, inplace=True)
    return df_usd.drop_duplicates(subset=['Data'], keep='last')

def extrair_brent_yfinance():
    ticker = yf.Ticker("BZ=F")
    df_brent = ticker.history(period="1mo").reset_index()
    df_brent = df_brent[['Date', 'Close']].copy()
    df_brent.rename(columns={'Date': 'Data', 'Close': 'Preco_Brent_USD'}, inplace=True)
    df_brent['Data'] = pd.to_datetime(df_brent['Data']).dt.tz_localize(None).dt.normalize()
    return df_brent

def autenticar_bigquery():
    try:
        user_secrets = UserSecretsClient()
        credenciais_json_str = user_secrets.get_secret("gcp_bq_json")
        credenciais_dict = json.loads(credenciais_json_str)
        credentials = service_account.Credentials.from_service_account_info(credenciais_dict)
        return bigquery.Client(credentials=credentials, project=credenciais_dict['project_id'])
    except Exception as e:
        print(f"Erro de autenticação no GCP: {e}")
        return None

def carregar_dados_bigquery(df, client, project_id, dataset_id, table_id):
    tabela_destino = f"{project_id}.{dataset_id}.{table_id}"
    
    query_max_data = f"SELECT MAX(Data) as ultima_data FROM `{tabela_destino}`"
    
    try:
        resultado = client.query(query_max_data).result()
        ultima_data_bq = None
        for row in resultado:
            ultima_data_bq = row.ultima_data
            
        if ultima_data_bq:
            ultima_data_bq = pd.to_datetime(ultima_data_bq).tz_localize(None)
            print(f"Última data encontrada no BigQuery: {ultima_data_bq.date()}")
            
            df = df[df['Data'] > ultima_data_bq]
        else:
            print("Tabela vazia no BigQuery. Iniciando primeira carga total.")
            
    except Exception as e:
        print(f"Aviso ao consultar tabela (normal em primeira execução ou tabela inexistente): {e}")

    if df.empty:
        print("Nenhum dado novo para carregar hoje. A integridade foi mantida.")
        return

    print(f"Preparando inserção de {len(df)} novas linhas...")
    
    # Insere fazendo Append
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
    job = client.load_table_from_dataframe(df, tabela_destino, job_config=job_config)
    job.result()
    
    print("✅ Carga concluída com sucesso no BigQuery!")


def executar_pipeline_fase3():
    PROJECT_ID = "pipeline-brent-bcb" 
    DATASET_ID = "dados_financeiros"
    TABLE_ID = "paridade_brent"

    print("1. Extraindo e Transformando Dados...")
    df_usd = extrair_dolar_bcb_olinda()
    df_brent = extrair_brent_yfinance()
    df_final = pd.merge(df_brent, df_usd, on='Data', how='inner')
    df_final['Preco_Brent_BRL'] = (df_final['Preco_Brent_USD'] * df_final['Cotacao_USD_BRL']).round(2)
    df_final['Preco_Brent_USD'] = df_final['Preco_Brent_USD'].round(2)
    df_final.sort_values('Data', inplace=True)
    
    print("2. Autenticando no BigQuery...")
    bq_client = autenticar_bigquery()
    if not bq_client: return
    
    print("3. Executando Job de Carga...")
    carregar_dados_bigquery(df_final, bq_client, PROJECT_ID, DATASET_ID, TABLE_ID)

if __name__ == "__main__":
    executar_pipeline_fase3()

1. Extraindo e Transformando Dados...
2. Autenticando no BigQuery...
3. Executando Job de Carga...
Última data encontrada no BigQuery: 2026-09-21
Preparando inserção de 1 novas linhas...
✅ Carga concluída com sucesso no BigQuery!
